# Chapter 08: Coordinates

Source orientation: printed pages 107-134; PDF pages 125-152. I inspected the scanned source span privately and used it only as a map of topics and notation. This notebook is original: it does not copy textbook prose, exercise text, figures, screenshots, or page crops.

## Chapter Goal

Coordinates are not just a way to name points. They are a translation device between synthetic geometry and computable invariants. In this chapter we use coordinates to move from points on a line, to Cartesian and polar descriptions in the plane, to conics and circle pencils, to arc length and area, to hyperbolic functions, to equiangular spirals, and finally to three-dimensional equations. The central question is: when a geometric claim becomes an equation, what does the equation make visible that a static drawing can hide?

## Source-Oriented Storyboard

The source span begins with Cartesian coordinates and treats plane curves through equations. It then shifts between rectangular and polar language, uses circles and conics as coordinate laboratories, derives tangent, arc length, and area formulas, compares circular with hyperbolic functions, studies the equiangular spiral as a curve whose shape survives scale-and-rotate motions, and closes with three-dimensional coordinates, lines, planes, spheres, and ruled quadrics.

The visual route here is direct rather than decorative:

- a coordinate dictionary that shows Cartesian, polar, circle, conic, and tangent data on the same stage;
- a conic classifier that turns a quadratic equation into ellipse/parabola/hyperbola behavior and checks a tangent condition;
- an arc-length and area lab that compares polygonal approximations with integral formulas;
- a hyperbolic-function panel that makes the circle/hyperbola sector analogy inspectable;
- an equiangular spiral experiment that checks the constant tangent angle and self-similarity under rotation plus dilation;
- a 3D coordinate view of lines, planes, a sphere tangent plane, and the two ruling families of a hyperbolic paraboloid.

## Translation Guide

A point becomes a tuple; a line becomes an equation or a parametric path; a circle or conic becomes a level set; a tangent becomes a derivative or a gradient condition; length and area become integrals; a spiral becomes a curve whose radial growth is tied to angular change; a 3D object becomes a constraint in three variables. The notebook keeps those translations visible by pairing every figure with a JSON or CSV check.


## How To Read The Coordinate Artifacts

The chapter deliberately repeats one pattern: choose a coordinate language, ask what it makes easy to see, and then check the claim in a second representation. In the coordinate dictionary, the same plane is asked to hold rectangular points, polar rays, conic level sets, and tangent data. That is not a catalog for its own sake. It shows why the coordinate choice belongs to the geometry: a circle centered at the origin welcomes polar coordinates, a conic classifier wants the quadratic coefficients, and a tangent calculation wants either a derivative along a parameter or a gradient perpendicular to the level set.

The later sections use the same discipline. The arc-length artifact compares the visible polygonal approximation with an independent perimeter estimate, while the area calculation uses the determinant form that records how a moving radius sweeps the plane. The hyperbolic-function panel asks the reader to compare two similar-looking parameterizations and notice the metric sign change: the circular identity protects `x^2 + y^2`, while the hyperbolic identity protects `x^2 - y^2`. The spiral lab is a stress test for polar coordinates because rotating the parameter is inseparable from scaling the radius. The 3D artifact then raises the stakes: equations for planes, spheres, and ruled quadrics are compact, but the interactive view is what lets a learner inspect the line families and tangent-plane condition. Each figure therefore has a nearby invariant so the notebook remains a course chapter, not a gallery.


In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, math, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import Markdown, display
import plotly.graph_objects as go
import sympy as sp

CHAPTER_NO = 8
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

chapter = chapter_by_no(CHAPTER_NO)
ARTIFACT_DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = ARTIFACT_DIRS["figures"]
HTML = ARTIFACT_DIRS["html"]
CHECKS = ARTIFACT_DIRS["checks"]
TABLES = ARTIFACT_DIRS["tables"]
DATA = ARTIFACT_DIRS["data"]

# Remove the old generic scaffold artifacts if this notebook is re-run after a bootstrap pass.
for stale in [
    FIG / "concept_configuration.svg",
    FIG / "parameter_experiment.svg",
    TABLES / "artifact_manifest.csv",
]:
    if stale.exists():
        stale.unlink()

artifact_records = []
def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def remember(path: Path, kind: str, concept: str, representation: str, inspection: str):
    artifact_records.append({
        "artifact": rel(path),
        "kind": kind,
        "concept": concept,
        "representation": representation,
        "inspection_target": inspection,
    })
    return path

def savefig(path: Path, fig, concept: str, inspection: str, dpi: int = 170):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    remember(path, "figure", concept, "Matplotlib PNG", inspection)
    return path

def display_many(paths):
    for path in paths:
        display(Markdown(f"### {Path(path).stem.replace('-', ' ').replace('_', ' ')}"))
        display_artifact(path, width=820)

display(Markdown(f"Loaded **{chapter['title']}**. Artifacts will be written under `{rel(BOOK_ROOT / 'artifacts' / f'chapter-{CHAPTER_NO:02d}')}`."))


## 1. Coordinate Dictionary: One Curve, Several Languages

The chapter’s first move is psychological as much as technical: a geometric object can be recognized by ordered numbers, by an equation, by a parameter, or by a polar rule. The figure below deliberately puts several languages next to one another. A circle shows the rectangular equation `x^2 + y^2 = r^2`; an ellipse and hyperbola show how coefficients change shape; a polar spiral shows that `(r, theta)` can be more natural than `(x, y)`; and tangent vectors mark how calculus enters the coordinate story.


In [ ]:
# Coordinate dictionary and conic/tangent checks.
theta = np.linspace(0, 2*np.pi, 500)
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
colors = {"blue": "#2563eb", "orange": "#d97706", "green": "#15803d", "red": "#b91c1c", "ink": "#111827"}

# Cartesian circle and polar rays.
ax = axs[0, 0]
r = 1.0
ax.plot(r*np.cos(theta), r*np.sin(theta), color=colors["blue"], lw=2.5, label=r"$x^2+y^2=1$")
for ang in np.linspace(0, 2*np.pi, 12, endpoint=False):
    ax.plot([0, np.cos(ang)], [0, np.sin(ang)], color="#cbd5e1", lw=0.8)
point_angle = 0.75
P = np.array([np.cos(point_angle), np.sin(point_angle)])
ax.scatter([P[0]], [P[1]], color=colors["red"], zorder=3)
ax.plot([0, P[0]], [0, P[1]], color=colors["red"], lw=1.5)
ax.text(P[0]+0.05, P[1], r"$(r,\theta)$", fontsize=11)
ax.set_title("Cartesian circle with polar address")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb"); ax.legend(loc="lower left")

# Conics from quadratic equations.
ax = axs[0, 1]
x = np.linspace(-2.4, 2.4, 500)
ell_y = 0.65*np.sqrt(np.maximum(0, 1 - (x/1.7)**2))
ax.plot(x, ell_y, color=colors["blue"], lw=2, label="ellipse")
ax.plot(x, -ell_y, color=colors["blue"], lw=2)
par_y = 0.35*x**2 - 1.1
ax.plot(x, par_y, color=colors["green"], lw=2, label="parabola")
hx = np.linspace(0.75, 2.4, 300)
hy = 0.7*np.sqrt((hx/0.8)**2 - 1)
ax.plot(hx, hy, color=colors["orange"], lw=2, label="hyperbola")
ax.plot(-hx, hy, color=colors["orange"], lw=2)
ax.plot(hx, -hy, color=colors["orange"], lw=2)
ax.plot(-hx, -hy, color=colors["orange"], lw=2)
ax.set_title("Quadratic equations separate conic types")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb"); ax.legend(loc="upper right")

# Tangent as derivative/gradient.
ax = axs[1, 0]
a, b = 1.8, 1.0
ax.plot(a*np.cos(theta), b*np.sin(theta), color=colors["blue"], lw=2.5)
t0 = 0.9
Q = np.array([a*np.cos(t0), b*np.sin(t0)])
tangent = np.array([-a*np.sin(t0), b*np.cos(t0)])
tangent = tangent / np.linalg.norm(tangent)
normal = np.array([Q[0]/a**2, Q[1]/b**2])
normal = normal / np.linalg.norm(normal)
ax.scatter([Q[0]], [Q[1]], color=colors["red"], zorder=3)
ax.arrow(Q[0], Q[1], 0.75*tangent[0], 0.75*tangent[1], head_width=0.05, color=colors["green"], length_includes_head=True)
ax.arrow(Q[0], Q[1], 0.55*normal[0], 0.55*normal[1], head_width=0.05, color=colors["red"], length_includes_head=True)
ax.text(Q[0]+0.08, Q[1]+0.08, "tangent vs gradient", fontsize=10)
ax.set_title("Tangent is perpendicular to the gradient")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb")

# Polar curve.
ax = axs[1, 1]
ang = np.linspace(0, 5*np.pi, 700)
rho = 0.18*np.exp(0.12*ang)
ax.plot(rho*np.cos(ang), rho*np.sin(ang), color=colors["orange"], lw=2.5)
for k in range(0, len(ang), 120):
    ax.plot([0, rho[k]*np.cos(ang[k])], [0, rho[k]*np.sin(ang[k])], color="#cbd5e1", lw=0.7)
ax.set_title(r"Polar curve $r = ae^{b\theta}$")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb")

for ax in axs.flat:
    ax.axhline(0, color="#94a3b8", lw=0.8)
    ax.axvline(0, color="#94a3b8", lw=0.8)

coordinate_dictionary = savefig(FIG / "coordinate-dictionary-conics-tangents.png", fig, "Coordinate systems and conic/tangent dictionary", "Compare Cartesian, polar, conic, and tangent representations on one coordinate stage")

A, B, C = sp.symbols("A B C")
# For Ax^2 + Cy^2 + ... the sign of B^2-4AC classifies the centered conic type.
conic_checks = {
    "ellipse_discriminant": int(0**2 - 4*1*4),
    "parabola_discriminant": int(0**2 - 4*1*0),
    "hyperbola_discriminant": int(0**2 - 4*1*(-1)),
    "ellipse_tangent_dot_gradient": float(np.dot(tangent, normal)),
}
assert conic_checks["ellipse_discriminant"] < 0
assert conic_checks["parabola_discriminant"] == 0
assert conic_checks["hyperbola_discriminant"] > 0
assert abs(conic_checks["ellipse_tangent_dot_gradient"]) < 1e-12
conic_path = write_json(CHECKS / "coordinate-dictionary-checks.json", conic_checks)
remember(conic_path, "check", "Conic classification and tangent condition", "JSON", "Check conic discriminant signs and tangent-gradient perpendicularity")

display_many([coordinate_dictionary])
conic_checks


## 2. Arc Length And Area Are Coordinates With Memory

Coordinates also let us measure a curve by remembering how it is traversed. For a parametric curve `(x(t), y(t))`, small secant steps approximate `ds`; the limit becomes the arc-length integral. For area, the determinant-like expression `x dy - y dx` records swept sector area. The table and figure below use an ellipse because it is simple enough to compute and curved enough that the polygonal approximation visibly converges.


In [ ]:
# Arc length and area approximation for an ellipse.
a, b = 2.0, 1.15
sample_counts = [24, 48, 96, 192, 384]
rows = []
for n in sample_counts:
    t = np.linspace(0, 2*np.pi, n+1)
    pts = np.column_stack([a*np.cos(t), b*np.sin(t)])
    segs = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    polygon_length = float(segs.sum())
    shoelace_area = 0.5*abs(np.dot(pts[:-1,0], pts[1:,1]) - np.dot(pts[1:,0], pts[:-1,1]))
    rows.append({"samples": n, "polygon_length": polygon_length, "shoelace_area": float(shoelace_area), "area_error": float(abs(shoelace_area - math.pi*a*b))})
arc_df = pd.DataFrame(rows)
arc_table = write_csv(TABLES / "ellipse-arc-area-convergence.csv", rows)
remember(arc_table, "table", "Ellipse arc length and area convergence", "CSV", "Inspect convergence of polygonal length and determinant-style area")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 5))
t = np.linspace(0, 2*np.pi, 500)
ax0.plot(a*np.cos(t), b*np.sin(t), color=colors["blue"], lw=2.5, label="ellipse")
coarse = np.linspace(0, 2*np.pi, 17)
coarse_pts = np.column_stack([a*np.cos(coarse), b*np.sin(coarse)])
ax0.plot(coarse_pts[:,0], coarse_pts[:,1], "o-", color=colors["orange"], lw=1.4, ms=3.5, label="polygonal ds")
# tangent sample
u = 1.25
P = np.array([a*np.cos(u), b*np.sin(u)])
V = np.array([-a*np.sin(u), b*np.cos(u)])
V = V / np.linalg.norm(V)
ax0.arrow(P[0], P[1], 0.65*V[0], 0.65*V[1], head_width=0.06, color=colors["red"], length_includes_head=True)
ax0.set_title("Arc length from secant steps")
ax0.set_aspect("equal"); ax0.grid(True, color="#e5e7eb"); ax0.legend()

ax1.plot(arc_df["samples"], arc_df["area_error"], "o-", color=colors["green"], label="area error")
ax1.set_xscale("log", base=2); ax1.set_yscale("log")
ax1.set_xlabel("sample count"); ax1.set_ylabel("absolute area error")
ax1.set_title("Determinant area converges")
ax1.grid(True, color="#e5e7eb", which="both")
ax1.legend()
arc_fig = savefig(FIG / "arc-length-area-convergence.png", fig, "Arc length and area convergence", "Compare secant-length approximation with determinant area convergence")

# Ramanujan approximation gives a compact independent length check for the ellipse.
h = ((a-b)**2)/((a+b)**2)
ramanujan = math.pi*(a+b)*(1 + 3*h/(10 + math.sqrt(4 - 3*h)))
arc_checks = {
    "ellipse_area_exact": math.pi*a*b,
    "last_area_error": float(arc_df.iloc[-1]["area_error"]),
    "ramanujan_perimeter_estimate": ramanujan,
    "last_polygon_length": float(arc_df.iloc[-1]["polygon_length"]),
    "relative_length_gap_to_ramanujan": float(abs(arc_df.iloc[-1]["polygon_length"] - ramanujan)/ramanujan),
}
assert arc_checks["last_area_error"] < arc_df.iloc[0]["area_error"]
assert arc_checks["relative_length_gap_to_ramanujan"] < 3e-4
arc_check_path = write_json(CHECKS / "arc-length-area-checks.json", arc_checks)
remember(arc_check_path, "check", "Arc length and area numerical invariants", "JSON", "Confirm area convergence and perimeter sanity against an independent approximation")

display_many([arc_fig])
arc_df.tail()


## 3. Circular And Hyperbolic Functions

The source span compares the unit circle with the rectangular hyperbola. In the circular case, `(cos t, sin t)` has `x^2 + y^2 = 1`; in the hyperbolic case, `(cosh t, sinh t)` has `x^2 - y^2 = 1`. The analogy is not a slogan: both parameterizations turn a geometric sector into a measured parameter, and both satisfy a differential system that swaps the coordinates with a sign change.


In [ ]:
# Circle/hyperbola comparison with exact symbolic checks.
t = np.linspace(-1.4, 1.4, 400)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].plot(np.cos(np.linspace(0, 2*np.pi, 500)), np.sin(np.linspace(0, 2*np.pi, 500)), color=colors["blue"], lw=2.5)
for val in [0.35, 0.9, 1.25]:
    axs[0].plot([0, np.cos(val)], [0, np.sin(val)], color=colors["orange"], lw=1.3)
    axs[0].scatter([np.cos(val)], [np.sin(val)], color=colors["red"], s=25)
axs[0].set_title(r"Circle: $(\cos t, \sin t)$")
axs[0].set_aspect("equal"); axs[0].grid(True, color="#e5e7eb")

axs[1].plot(np.cosh(t), np.sinh(t), color=colors["green"], lw=2.5, label="right branch")
axs[1].plot(-np.cosh(t), np.sinh(t), color=colors["green"], lw=1.5, alpha=0.45)
for val in [0.35, 0.9, 1.25]:
    axs[1].plot([0, np.cosh(val)], [0, np.sinh(val)], color=colors["orange"], lw=1.3)
    axs[1].scatter([np.cosh(val)], [np.sinh(val)], color=colors["red"], s=25)
xx = np.linspace(1, 2.4, 300)
axs[1].plot(xx, np.sqrt(xx**2-1), color="#94a3b8", lw=0.8)
axs[1].plot(xx, -np.sqrt(xx**2-1), color="#94a3b8", lw=0.8)
axs[1].set_title(r"Rectangular hyperbola: $(\cosh t, \sinh t)$")
axs[1].set_aspect("equal"); axs[1].grid(True, color="#e5e7eb")

hyperbolic_fig = savefig(FIG / "circular-hyperbolic-sector-analogy.png", fig, "Circular and hyperbolic coordinate analogy", "Inspect how circle and rectangular-hyperbola parameters encode sector geometry")

u = sp.symbols("u")
symbolic_checks = {
    "cos_sin_identity": str(sp.simplify(sp.cos(u)**2 + sp.sin(u)**2)),
    "cosh_sinh_identity": str(sp.simplify(sp.cosh(u)**2 - sp.sinh(u)**2)),
    "d_cos": str(sp.diff(sp.cos(u), u)),
    "d_sinh": str(sp.diff(sp.sinh(u), u)),
}
assert symbolic_checks["cos_sin_identity"] == "1"
assert symbolic_checks["cosh_sinh_identity"] == "1"
hyperbolic_path = write_json(CHECKS / "circular-hyperbolic-identities.json", symbolic_checks)
remember(hyperbolic_path, "check", "Circular and hyperbolic function identities", "JSON", "Verify the algebraic identities and derivative swaps behind the visual analogy")

display_many([hyperbolic_fig])
symbolic_checks


## 4. Equiangular Spiral: A Curve That Survives Its Own Scaling

The equiangular, or logarithmic, spiral is a perfect coordinate lesson because the equation is the behavior: `r = a exp(b theta)`. Rotating the parameter by `delta` multiplies the radius by `exp(b delta)`, so the curve returns to the same shape at a different scale. Its tangent keeps a constant angle with the radius vector. That is a geometric invariant, not a decoration.


In [ ]:
# Equiangular spiral constant-angle and self-similarity experiment.
b = 0.18
ang = np.linspace(0.0, 7*np.pi, 900)
rho = 0.13*np.exp(b*ang)
x = rho*np.cos(ang); y = rho*np.sin(ang)
# Analytic tangent vector for r=a exp(b theta): d/dtheta [r cos theta, r sin theta]
dx = rho*(b*np.cos(ang) - np.sin(ang))
dy = rho*(b*np.sin(ang) + np.cos(ang))
radial = np.column_stack([np.cos(ang), np.sin(ang)])
tangent = np.column_stack([dx, dy])
cos_angles = np.einsum("ij,ij->i", radial, tangent) / np.linalg.norm(tangent, axis=1)
constant_angle = np.degrees(np.arccos(np.clip(cos_angles, -1, 1)))
expected_angle = np.degrees(math.atan2(1.0, b))

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(x, y, color=colors["orange"], lw=2.4, label=r"$r=a e^{b\theta}$")
for idx in np.linspace(80, 780, 7, dtype=int):
    p = np.array([x[idx], y[idx]])
    rv = radial[idx]
    tv = tangent[idx] / np.linalg.norm(tangent[idx])
    ax.plot([0, p[0]], [0, p[1]], color="#cbd5e1", lw=0.8)
    ax.arrow(p[0], p[1], 0.24*tv[0], 0.24*tv[1], color=colors["red"], head_width=0.025, length_includes_head=True)
ax.set_title("Equiangular spiral: constant tangent angle")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb"); ax.legend()
spiral_fig = savefig(FIG / "equiangular-spiral-constant-angle.png", fig, "Equiangular spiral tangent invariant", "Inspect constant angle between radius and tangent along a logarithmic spiral")

# Plotly HTML self-similarity lab.
delta_values = np.array([0.0, math.pi/2, math.pi, 3*math.pi/2])
fig_html = go.Figure()
for delta in delta_values:
    scale = math.exp(b*delta)
    xs = scale * x * math.cos(delta) - scale * y * math.sin(delta)
    ys = scale * x * math.sin(delta) + scale * y * math.cos(delta)
    fig_html.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=f"rotate {delta:.2f}, scale {scale:.2f}"))
fig_html.update_layout(
    title="Equiangular spiral self-similarity under rotate+dilate",
    xaxis=dict(scaleanchor="y", scaleratio=1, zeroline=True),
    yaxis=dict(zeroline=True),
    template="plotly_white",
    width=820,
    height=620,
)
spiral_html = HTML / "equiangular-spiral-self-similarity.html"
fig_html.write_html(spiral_html, include_plotlyjs="cdn", full_html=True)
remember(spiral_html, "html", "Equiangular spiral self-similarity", "Plotly HTML", "Toggle rotated and dilated copies to see that the curve shape persists")

spiral_checks = {
    "b": b,
    "expected_tangent_angle_degrees": expected_angle,
    "measured_angle_min_degrees": float(constant_angle.min()),
    "measured_angle_max_degrees": float(constant_angle.max()),
    "self_similarity_scales": [float(math.exp(b*d)) for d in delta_values],
    "max_angle_spread_degrees": float(constant_angle.max() - constant_angle.min()),
}
assert spiral_checks["max_angle_spread_degrees"] < 1e-10
spiral_check_path = write_json(CHECKS / "equiangular-spiral-checks.json", spiral_checks)
remember(spiral_check_path, "check", "Equiangular spiral invariants", "JSON", "Check constant tangent angle and rotate-dilate scale factors")

display_many([spiral_fig, spiral_html])
spiral_checks


## 5. Three-Dimensional Coordinates And Ruled Quadrics

The final source pages move into three dimensions. A line can be a parametric equation, a plane can be a linear equation, a sphere can be a quadratic equation, and a tangent plane can be read from a gradient. The closing exercises point toward quadrics such as the hyperbolic paraboloid, whose two ruling families make a surface out of straight lines. The interactive artifact below shows why coordinates are powerful here: the formula is compact, while the surface behavior is spatial.


In [ ]:
# 3D coordinates, tangent plane to a sphere, and hyperbolic paraboloid ruling families.
# Sphere x^2+y^2+z^2=R^2 at point P has tangent plane P dot X = R^2.
R = 2.0
P = np.array([1.0, 1.0, math.sqrt(R**2 - 2.0)])
plane_residual_at_P = float(np.dot(P, P) - R**2)

u = np.linspace(-2, 2, 21)
v = np.linspace(-2, 2, 21)
U, V = np.meshgrid(u, v)
Z = 0.25*(U**2 - V**2)

fig3 = go.Figure()
fig3.add_trace(go.Surface(x=U, y=V, z=Z, colorscale="RdBu", opacity=0.64, showscale=False, name="z=(x^2-y^2)/4"))
# Two ruling families: x=s+t, y=s-t, z=st and x=s+t, y=-s+t, z=-st scaled for z=(x^2-y^2)/4.
tvals = np.linspace(-2.2, 2.2, 80)
for s in np.linspace(-1.6, 1.6, 7):
    x1 = s + tvals; y1 = s - tvals; z1 = s*tvals
    x2 = s + tvals; y2 = -s + tvals; z2 = -s*tvals
    fig3.add_trace(go.Scatter3d(x=x1, y=y1, z=z1, mode="lines", line=dict(color="#d97706", width=4), showlegend=False))
    fig3.add_trace(go.Scatter3d(x=x2, y=y2, z=z2, mode="lines", line=dict(color="#2563eb", width=4), showlegend=False))
# Sphere point and tangent plane patch.
alpha = np.linspace(-0.8, 0.8, 5)
beta = np.linspace(-0.8, 0.8, 5)
A, B = np.meshgrid(alpha, beta)
basis1 = np.array([1.0, -1.0, 0.0]); basis1 /= np.linalg.norm(basis1)
basis2 = np.cross(P, basis1); basis2 /= np.linalg.norm(basis2)
plane_pts = P + A[...,None]*basis1 + B[...,None]*basis2
fig3.add_trace(go.Surface(x=plane_pts[...,0], y=plane_pts[...,1], z=plane_pts[...,2], opacity=0.38, colorscale=[[0,"#bbf7d0"],[1,"#bbf7d0"]], showscale=False, name="sphere tangent plane"))
fig3.add_trace(go.Scatter3d(x=[0, P[0]], y=[0, P[1]], z=[0, P[2]], mode="lines+markers", line=dict(color="#111827", width=5), marker=dict(size=4), name="sphere radius to tangent point"))
fig3.update_layout(
    title="3D coordinates: ruled hyperbolic paraboloid and sphere tangent plane",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    template="plotly_white",
    width=900,
    height=720,
)
quadric_html = HTML / "three-dimensional-coordinates-ruled-quadric.html"
fig3.write_html(quadric_html, include_plotlyjs="cdn", full_html=True)
remember(quadric_html, "html", "3D coordinates and ruled quadrics", "Plotly 3D HTML", "Rotate the ruled surface and compare its straight-line families with the tangent plane construction")

# Static projection summary for validators/readers who do not open HTML.
fig, ax = plt.subplots(figsize=(8, 6))
for s in np.linspace(-1.6, 1.6, 7):
    x1 = s + tvals; y1 = s - tvals
    x2 = s + tvals; y2 = -s + tvals
    ax.plot(x1, y1, color=colors["orange"], lw=1.2)
    ax.plot(x2, y2, color=colors["blue"], lw=1.2)
ax.scatter([P[0]], [P[1]], color=colors["red"], label="sphere tangent point projection")
ax.set_title("Two ruling families projected to the xy-plane")
ax.set_aspect("equal"); ax.grid(True, color="#e5e7eb"); ax.legend()
quadric_png = savefig(FIG / "ruled-quadric-families-projection.png", fig, "Ruled quadric line families", "Inspect two straight-line families whose coordinates sweep a hyperbolic paraboloid")

quadric_checks = {
    "sphere_radius_squared": R**2,
    "tangent_point_norm_squared": float(np.dot(P, P)),
    "plane_residual_at_tangent_point": plane_residual_at_P,
    "ruled_surface_family_count": 2,
    "sample_lines_per_family": 7,
}
assert abs(quadric_checks["plane_residual_at_tangent_point"]) < 1e-12
quadric_path = write_json(CHECKS / "three-dimensional-coordinate-checks.json", quadric_checks)
remember(quadric_path, "check", "3D coordinate and quadric invariants", "JSON", "Check sphere tangent plane residual and ruled-surface family counts")

display_many([quadric_png, quadric_html])
quadric_checks


## Applied Lab: Classify A Quadratic And Choose Coordinates

The lab below asks a practical question: given a quadratic equation, which coordinate view should you use? The discriminant `B^2 - 4AC` tells whether the centered conic behaves like an ellipse, parabola, or hyperbola. That does not solve every geometric question, but it tells the reader where to look: closed oval, single bending branch, or two opposing branches. This is a small example of the chapter’s larger lesson: coordinates compress a geometric decision into a computable invariant.


In [ ]:
quadratics = [
    {"name": "circle", "A": 1, "B": 0, "C": 1, "coordinate_view": "Cartesian or polar"},
    {"name": "ellipse", "A": 4, "B": 0, "C": 9, "coordinate_view": "Cartesian axes scaled by semi-axes"},
    {"name": "parabola", "A": 1, "B": 0, "C": 0, "coordinate_view": "axis plus tangent direction"},
    {"name": "hyperbola", "A": 1, "B": 0, "C": -1, "coordinate_view": "asymptote frame or hyperbolic parameter"},
    {"name": "rotated hyperbola", "A": 1, "B": 3, "C": 1, "coordinate_view": "rotate axes to principal directions"},
]
for row in quadratics:
    disc = row["B"]**2 - 4*row["A"]*row["C"]
    row["discriminant"] = disc
    row["type"] = "parabolic" if disc == 0 else "elliptic" if disc < 0 else "hyperbolic"
quad_df = pd.DataFrame(quadratics)
quad_table = write_csv(TABLES / "quadratic-coordinate-view-lab.csv", quadratics)
remember(quad_table, "table", "Quadratic conic classification lab", "CSV", "Map conic discriminants to the coordinate view that makes the geometry easiest to inspect")

lab_checks = {
    "types": dict(zip(quad_df["name"], quad_df["type"])),
    "all_expected_types_present": sorted(quad_df["type"].unique().tolist()),
}
assert lab_checks["types"]["circle"] == "elliptic"
assert lab_checks["types"]["parabola"] == "parabolic"
assert lab_checks["types"]["hyperbola"] == "hyperbolic"
lab_path = write_json(CHECKS / "quadratic-coordinate-view-lab.json", lab_checks)
remember(lab_path, "check", "Quadratic coordinate-view lab checks", "JSON", "Confirm discriminant classification for representative coordinate choices")
quad_df


## Takeaways

- Coordinates turn geometric objects into addressable data: tuples, equations, parameters, gradients, and constraints.
- Cartesian and polar descriptions are not rivals; each is chosen because it makes a different invariant easier to inspect.
- Conic classification, tangent conditions, arc length, and swept area all become checks that can fail if the model is wrong.
- Hyperbolic functions mirror circular functions through a changed metric sign: `x^2 + y^2` becomes `x^2 - y^2`.
- The equiangular spiral is a coordinate equation with a visible invariant: its tangent keeps a constant angle with the radius while rotation and dilation reproduce the same shape.
- In three dimensions, coordinates make lines, tangent planes, spheres, and ruled quadrics inspectable by equations plus interactive views.


In [ ]:
# Final chapter-specific sanity checks and artifact manifest.
manifest_csv = TABLES / "artifact-manifest.csv"
remember(manifest_csv, "table", "Chapter artifact manifest", "CSV", "List every generated chapter-08 artifact and its inspection target")
write_csv(manifest_csv, artifact_records)

visual_summary = {
    "chapter": CHAPTER_NO,
    "title": chapter["title"],
    "source_span": {"printed_pages": chapter["printed"], "pdf_pages": chapter["pdf"]},
    "figures": [row["artifact"] for row in artifact_records if row["kind"] == "figure"],
    "html": [row["artifact"] for row in artifact_records if row["kind"] == "html"],
    "tables": [row["artifact"] for row in artifact_records if row["kind"] == "table"],
    "checks": [row["artifact"] for row in artifact_records if row["kind"] == "check"],
    "storyboard": [
        "coordinate dictionary across Cartesian, polar, conic, and tangent representations",
        "arc length and determinant-area convergence",
        "circular versus hyperbolic function identities",
        "equiangular spiral tangent angle and rotate-dilate self-similarity",
        "3D coordinate tangent plane and ruled quadric families",
        "quadratic discriminant lab",
    ],
}
visual_summary_path = CHECKS / "visual_summary.json"
write_json(visual_summary_path, visual_summary)
remember(visual_summary_path, "check", "Visual summary", "JSON", "Audit source span, artifact paths, and chapter-specific storyboard coverage")

all_artifacts = [BOOK_ROOT / row["artifact"] for row in artifact_records]
assert_artifacts(all_artifacts, min_bytes=100)
assert len(visual_summary["figures"]) >= 4
assert len(visual_summary["html"]) >= 2
assert len(visual_summary["checks"]) >= 6
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()

final_sanity = {
    "artifact_count": len(all_artifacts),
    "visual_count": len(visual_summary["figures"]) + len(visual_summary["html"]),
    "check_count": len(visual_summary["checks"]),
    "manifest": rel(manifest_csv),
    "visual_summary": rel(visual_summary_path),
    "artifact_sizes": {rel(path): path.stat().st_size for path in all_artifacts},
}
final_sanity
